# 42 — Tool Calling & Agent Loops

### Why this exists
An agent is a control loop with authority. This lab separates tool selection, argument validation, authorization, execution and termination.

### Learning objectives
model tool calls; validate arguments; enforce permissions; cap loops

### Prerequisite
Python dictionaries and state machines

### Engineering loop
**Predict → Build → Observe → Break → Diagnose → Improve → Generalize → Defend**

## 🖊️ Sketch note
![Sketch note](../assets/sketch-notes/42-tool-calling-agent-loops.svg)

> **Question to carry through the lab:** Where is authority enforced: in the prompt, the tool, or the execution boundary?

## Capability contract

**Capability:** C08 — LLM Systems  
**Workstream:** AI Support Platform  
**Primary roles:** AI Engineer|ML Engineer  
**You will prove:** Build or inspect a grounded AI mechanism


## Simulation → reality bridge

**Real-system bridge:** Tool-calling agent runtimes

**What this simulation proves:** It isolates the mechanism under study so you can predict and measure it without infrastructure noise.

**What the simulation hides:** The notebook models deterministic tool selection and authorization. Real agents add model-generated arguments, prompt/data injection, tool failures, retries, identity, permissions and irreversible side effects.

**Transfer challenge:** Design an approval boundary for one tool and identify the evidence required before allowing an agent to invoke it automatically.

**Production boundary:** Do not describe the toy implementation as the production technology. Before shipping, identify the real-system evidence, operational controls, and failure modes that remain untested here.


## Mental model
A reliable agent treats model output as an untrusted proposal. Policy and authorization belong outside the model and before side effects.

## 1. Predict before you run
Write down what you expect and what evidence would prove you wrong. Do not change the experiment after seeing the result.

## 2. Build
The first cell creates the smallest system that can answer the question. The later cells change one boundary at a time.

## Agent-native lens

**Agent can do:** routine implementation or a first-pass analysis.

**You must understand:** the mechanism and its assumptions.

**You must verify:** use tests, measurements or adversarial data that could prove the output wrong.

**You must decide:** the trade-off, failure response or production boundary.


## Apprenticeship bridge

Use this notebook to prepare for the following Acme Commerce work scenario(s). The scenario gives you symptoms and constraints; it does **not** give you the diagnosis.

- **S21 — 1232: AI assistant wants to take actions without clear authorization** → [`../work_simulations/21_acme_commerce_tool_guardrails/README.md`](../work_simulations/21_acme_commerce_tool_guardrails/README.md)

**How to use the bridge:** finish the experiment and independent challenge here, then enter the ticket without reopening the notebook as an answer key. Bring your own prediction, evidence and verification plan.


In [ ]:
tools={
 "lookup_order":{"scope":"read","args":{"order_id":str}},
 "issue_refund":{"scope":"write","args":{"order_id":str,"amount":float}}
}
request={"tool":"issue_refund","args":{"order_id":"o1","amount":50.0}}
print(request)

In [ ]:
def validate_call(call):
    spec=tools.get(call.get("tool"));
    if not spec: return False,"unknown_tool"
    for k,t in spec["args"].items():
        if k not in call["args"] or not isinstance(call["args"][k],t): return False,f"bad_arg:{k}"
    return True,"ok"
print(validate_call(request))

In [ ]:
def authorize(call,user_scope="read"):
    ok,_=validate_call(call)
    if not ok: return False
    return tools[call["tool"]]["scope"]=="read" or user_scope=="write"
print("read-only actor",authorize(request)); print("write actor",authorize(request,"write"))

In [ ]:
# Break loop termination with a repeated tool request.
calls=[request,request,request,request]
max_actions=3
print("executed",min(len(calls),max_actions),"blocked",max(0,len(calls)-max_actions))

## 3. Measure
Do not stop at “it worked”. Record a correctness signal, a failure signal, and at least one quantitative measurement where the concept permits it.

## 4. Break deliberately
Introduce one controlled failure. Keep the failure reproducible so another engineer can reproduce the observation.

## 5. Diagnose
Use evidence, not the implementation you expected. State: **symptom → mechanism → evidence → boundary**. If two explanations fit, design one more measurement.

## 6. Improve
Make the smallest change that addresses the mechanism. Re-run the original experiment and the failure case.

## 7. Generalize
Ask what changes at 10× load, with retries, multiple writers/readers, partial failure, or changing contracts. Separate what this toy proves from what it only illustrates.

## 8. Production bridge
Production agents need identity, least privilege, tool schemas, action budgets, idempotency, traces, state freshness, human escalation and kill switches.

## 9. Independent challenge
Add stale state between two tool calls and a malformed amount. Decide whether the agent should retry, refresh state, escalate, or stop.

### Evidence to keep
argument validation, authorization decision, bounded action count, and side-effect policy

## 10. Explain it in 60–90 seconds
Explain the mechanism, your measurement, the failure you injected, the diagnosis, and the production decision you would make. Avoid tool names until the mental model is clear.